# SAR Oil Spill Detection — Colab Training Notebook

This notebook trains the U-Net model on the full dataset using a free Colab GPU.

## Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Put your dataset in Colab local storage under `/content/oil_spill_detection/FullDataset`
3. Run all cells from top to bottom

## Expected folder structure in Colab local storage:
```
/content/oil_spill_detection/FullDataset/
├── images/    ← all .tif SAR images
└── masks/     ← all .tif binary masks
```

You can still mount Google Drive later to save checkpoints/results.

---
## Cell 1 — Verify GPU
Always run this first. If it says CPU, go to Runtime → Change runtime type → T4 GPU.

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

---
## Cell 2 — (Optional) Mount Google Drive for Saving Results
You do not need Drive for dataset loading in this version.
Mount Drive only if you want checkpoints/results to persist after session reset.

In [ ]:
from google.colab import drive

MOUNT_DRIVE = True  # Set False if you do not want to use Drive at all

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    print('Drive mounted. You can save results to /content/drive/MyDrive/...')
else:
    print('Skipping Drive mount.')

---
## Cell 3 — Clone GitHub Repository
This pulls all your code (preprocess.py, dataset.py, model.py, train.py) from GitHub.

**Replace the URL with your actual GitHub repo URL.**

In [ ]:
import os

# ── Replace this with your actual GitHub repo URL ──
GITHUB_URL = 'https://github.com/YOUR_USERNAME/oil_spill_detection.git'
REPO_NAME  = 'oil_spill_detection'

# Clone or pull latest code
if os.path.exists(f'/content/{REPO_NAME}'):
    print('Repo already exists — pulling latest changes...')
    %cd /content/{REPO_NAME}
    !git pull origin main
else:
    print('Cloning repository...')
    !git clone {GITHUB_URL}
    %cd /content/{REPO_NAME}

print(f'\nWorking directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

---
## Cell 4 — Install Dependencies
Colab resets every session so we must reinstall packages each time.
This takes about 1-2 minutes.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations rasterio

# Verify installs
import segmentation_models_pytorch as smp
import albumentations as A
import rasterio
print(f'segmentation-models-pytorch : {smp.__version__}')
print(f'albumentations              : {A.__version__}')
print(f'rasterio                    : {rasterio.__version__}')
print('All packages installed successfully')

---
## Cell 5 — Verify Dataset in Colab Local Storage

This notebook reads the dataset directly from local Colab storage for faster I/O.

Expected location:
`/content/oil_spill_detection/FullDataset/images`
`/content/oil_spill_detection/FullDataset/masks`

If files are not there yet, download/copy them into this location first.

In [ ]:
import os
import shutil

LOCAL_DATA = '/content/oil_spill_detection/FullDataset'
IMAGES_DIR = f'{LOCAL_DATA}/images'
MASKS_DIR  = f'{LOCAL_DATA}/masks'

# Show available disk in /content
_, _, free = shutil.disk_usage('/content')
print(f'Free disk in /content: {free / (1024**3):.1f} GB')

if not os.path.exists(IMAGES_DIR) or not os.path.exists(MASKS_DIR):
    raise FileNotFoundError(
        'Dataset folder not found. Expected:\n'
        f'- {IMAGES_DIR}\n'
        f'- {MASKS_DIR}\n'
        'Upload/download the dataset to /content first.'
    )

images = [f for f in os.listdir(IMAGES_DIR) if f.endswith('.tif')]
masks  = [f for f in os.listdir(MASKS_DIR)  if f.endswith('.tif')]
print(f'Found {len(images)} images and {len(masks)} masks in local Colab storage')
print(f'Using dataset at: {LOCAL_DATA}')

---
## Cell 6 — Run Preprocessing

This runs `preprocess.py` on the full dataset to:
- Split images into train/val/test
- Compute normalization statistics
- Extract and save 256×256 patches

We use `stride=128` here (50% overlap) for more patches than local training.

**Note:** We override the paths to point to the full dataset.
This takes 5-15 minutes depending on dataset size.

In [ ]:
# Override config in preprocess.py for the full Colab dataset
import sys
sys.path.insert(0, '/content/oil_spill_detection')

import src.preprocess as pre

# Point to full dataset and use overlapping stride for more patches
pre.IMAGES_DIR  = 'FullDataset/images'
pre.MASKS_DIR   = 'FullDataset/masks'
pre.OUTPUT_DIR  = 'data/patches'
pre.STATS_FILE  = 'data/train_stats.json'
pre.STRIDE      = 128   # 50% overlap — more patches than local (stride=256)
pre.PATCH_SIZE  = 256

pre.run_preprocessing()
print('\nPreprocessing complete!')

---
## Cell 7 — Verify Patches
Quick check to confirm patches were created correctly before starting training.

In [ ]:
import os

for split in ['train', 'val', 'test']:
    split_dir = f'data/patches/{split}'
    if os.path.exists(split_dir):
        n_patches = len([f for f in os.listdir(split_dir) if f.endswith('_img.npy')])
        print(f'  {split:<6}: {n_patches} patches')
    else:
        print(f'  {split:<6}: NOT FOUND — preprocessing may have failed')

# Check stats file
import json
with open('data/train_stats.json') as f:
    stats = json.load(f)
print(f'\nNormalization stats:')
print(f'  mean: {stats["mean"]}')
print(f'  std : {stats["std"]}')

---
## Cell 8 — Train the Model

This runs the full training pipeline on GPU.

Settings used here:
- `--epochs 50` — enough for the model to converge on the full dataset
- `--batch_size 16` — larger than local (GPU has more memory than CPU RAM)
- `--lr 1e-4` — standard starting learning rate
- `--model_type smp` — pre-trained ResNet34 encoder

Expected time: **15-30 minutes** on a T4 GPU.

In [ ]:
!python src/train.py \
    --epochs 50 \
    --batch_size 16 \
    --lr 1e-4 \
    --model_type smp

---
## Cell 9 — Plot Training Curves
Visualize how loss and IoU evolved during training.
This helps you spot overfitting, underfitting, or a learning rate that's too high.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('checkpoints/training_log.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Curves', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(log['epoch'], log['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(log['epoch'], log['val_loss'],   label='Val Loss',   color='orange')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU
axes[1].plot(log['epoch'], log['train_iou'], label='Train IoU', color='steelblue')
axes[1].plot(log['epoch'], log['val_iou'],   label='Val IoU',   color='orange')
axes[1].set_title('IoU')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

best_epoch = log.loc[log['val_iou'].idxmax()]
print(f'Best epoch     : {int(best_epoch["epoch"])}')
print(f'Best val IoU   : {best_epoch["val_iou"]:.4f}')
print(f'Best val Dice  : {best_epoch["val_dice"]:.4f}')

---
## Cell 10 — Save Best Model to Persistent Storage

**IMPORTANT: Do this before your Colab session ends.**

Colab deletes everything in `/content/` when the session disconnects.
This cell saves artifacts to Google Drive (if mounted) so they persist.

In [ ]:
import shutil
import os

SAVE_DIR = '/content/drive/MyDrive/oil_spill_results'
if not os.path.exists('/content/drive/MyDrive'):
    raise RuntimeError('Google Drive is not mounted. Run Cell 2 first or change SAVE_DIR to a different persistent target.')

os.makedirs(SAVE_DIR, exist_ok=True)

# Save model checkpoint
shutil.copy('checkpoints/best_model.pth', f'{SAVE_DIR}/best_model.pth')
print('Saved: best_model.pth -> Drive')

# Save training log and curves
shutil.copy('checkpoints/training_log.csv', f'{SAVE_DIR}/training_log.csv')
shutil.copy('checkpoints/training_curves.png', f'{SAVE_DIR}/training_curves.png')
print('Saved: training_log.csv -> Drive')
print('Saved: training_curves.png -> Drive')

# Save normalization stats (needed for inference later)
shutil.copy('data/train_stats.json', f'{SAVE_DIR}/train_stats.json')
print('Saved: train_stats.json -> Drive')

print(f'\nAll files saved to: {SAVE_DIR}')

---
## Cell 11 — Commit Results to GitHub

We commit the training log and curves to GitHub so your progress is tracked.
We do NOT commit the .pth file (too large, already in .gitignore).

**Replace with your GitHub email and name.**

In [ ]:
import json
import pandas as pd

# Read best results
log      = pd.read_csv('checkpoints/training_log.csv')
best     = log.loc[log['val_iou'].idxmax()]
best_iou  = best['val_iou']
best_dice = best['val_dice']

# Configure git identity
!git config user.email "you@example.com"
!git config user.name  "Your Name"

# Stage files
!git add checkpoints/training_log.csv
!git add checkpoints/training_curves.png
!git add data/train_stats.json

# Commit with meaningful message
commit_msg = f'Colab training complete: val IoU={best_iou:.4f}, Dice={best_dice:.4f}'
!git commit -m "{commit_msg}"
!git push origin main

print(f'Pushed to GitHub: {commit_msg}')

---
## Notes

### If Colab disconnects mid-training
The `last_model.pth` checkpoint is saved after every epoch. To resume:
1. Re-run cells 1-7 to restore the environment
2. Modify `train.py` to load from `last_model.pth` before the training loop
3. Reduce `--epochs` by however many already completed

### If you run out of GPU memory
Reduce batch size: `--batch_size 8` instead of 16

### Upgrading the encoder for better accuracy
Replace `resnet34` with `efficientnet-b3` in `src/model.py` for better results at the cost of slower training.

### Next steps after training
1. Download `best_model.pth` from Drive back to your local machine
2. Run `src/evaluate.py` on the test set
3. Run `src/inference.py` on new images
4. Run `src/visualize.py` to generate geospatial maps